In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from bracket_functions import create_initial_bracket, JSONEncoder, save_brackets
from bracket_predictions import sequential_predictions, parallel_predictions
from constants import DATA_FILENAME, BRACKETS_JSON, BRACKETS_DIRECTORY, ORDER_OF_REGIONS, DATA_DIRECTORY
import json
import pandas as pd

In [3]:
pd.options.display.max_rows = 999

In [10]:
GENDER = "mens"

In [11]:
initial_bracket, data = create_initial_bracket(DATA_FILENAME.format(gender=GENDER), ORDER_OF_REGIONS[GENDER])

In [12]:
# 2023, 2024 values
sd_params = (11, 0)
k = 0

# 2022 values
# sd_params = (12, 1.9)
# k = 0

# 2021 values
# sd_params = (11, 0)
# sd_params = (10, -0.21)
# k = 0

In [13]:
%%time
# bracket_dict = sequential_predictions(11, initial_bracket, data, sd_params, k, score_method="torvik")
bracket_dict = sequential_predictions(11, initial_bracket, data, sd_params, k, score_method="kenpom")

CPU times: user 376 ms, sys: 41.9 ms, total: 418 ms
Wall time: 408 ms


In [14]:
# bracket_name = 'bracket8'
for bracket in range(0, 11):
    bracket_name = f'bracket{bracket}'
    bracket = bracket_dict[bracket_name][0]
    display(bracket.loc[bracket['8'] != ''][["Region", "Seed", "8", "4", "2", "1"]])

,Region,Seed,8,4,2,1
0,East,1,Duke,Duke,Duke,Duke
8,East,6,Louisville,,,
16,South,1,Florida,Florida,,
26,South,3,Illinois,,,
38,West,4,Arkansas,,,
42,West,3,Gonzaga,Gonzaga,Gonzaga,
48,Midwest,1,Michigan,Michigan,,
56,Midwest,6,Tennessee,,,


,Region,Seed,8,4,2,1
3,East,9,TCU,,,
10,East,3,Michigan St,Michigan St,Michigan St,
16,South,1,Florida,Florida,,
30,South,2,Houston,,,
35,West,9,Utah State,,,
46,West,2,Purdue,Purdue,,
48,Midwest,1,Michigan,Michigan,Michigan,Michigan
58,Midwest,3,Virginia,,,


,Region,Seed,8,4,2,1
0,East,1,Duke,Duke,,
14,East,2,UConn,,,
16,South,1,Florida,Florida,Florida,
24,South,6,North Carolina,,,
35,West,9,Utah State,,,
46,West,2,Purdue,Purdue,Purdue,Purdue
48,Midwest,1,Michigan,,,
56,Midwest,6,Tennessee,Tennessee,,


,Region,Seed,8,4,2,1
3,East,9,TCU,,,
10,East,3,Michigan St,Michigan St,Michigan St,Michigan St
16,South,1,Florida,,,
26,South,3,Illinois,Illinois,,
32,West,1,Arizona,Arizona,,
40,West,6,BYU,,,
48,Midwest,1,Michigan,,,
62,Midwest,2,Iowa State,Iowa State,Iowa State,


,Region,Seed,8,4,2,1
0,East,1,Duke,,,
14,East,2,UConn,UConn,,
22,South,4,Nebraska,,,
30,South,2,Houston,Houston,Houston,Houston
32,West,1,Arizona,,,
42,West,3,Gonzaga,Gonzaga,,
48,Midwest,1,Michigan,Michigan,Michigan,
58,Midwest,3,Virginia,,,


...

,Region,Seed,8,4,2,1
0,East,1,Duke,Duke,,
14,East,2,UConn,,,
20,South,5,Vanderbilt,Vanderbilt,Vanderbilt,
26,South,3,Illinois,,,
32,West,1,Arizona,,,
46,West,2,Purdue,Purdue,,
48,Midwest,1,Michigan,Michigan,Michigan,Michigan
61,Midwest,10,Santa Clara,,,


,Region,Seed,8,4,2,1
0,East,1,Duke,Duke,,
14,East,2,UConn,,,
20,South,5,Vanderbilt,,,
26,South,3,Illinois,Illinois,Illinois,Illinois
32,West,1,Arizona,Arizona,,
46,West,2,Purdue,,,
48,Midwest,1,Michigan,Michigan,Michigan,
56,Midwest,6,Tennessee,,,


,Region,Seed,8,4,2,1
0,East,1,Duke,Duke,Duke,Duke
10,East,3,Michigan St,,,
16,South,1,Florida,,,
30,South,2,Houston,Houston,,
32,West,1,Arizona,,,
40,West,6,BYU,BYU,,
48,Midwest,1,Michigan,Michigan,Michigan,
62,Midwest,2,Iowa State,,,


,Region,Seed,8,4,2,1
4,East,5,St John's,,,
8,East,6,Louisville,Louisville,,
22,South,4,Nebraska,Nebraska,Nebraska,
30,South,2,Houston,,,
32,West,1,Arizona,Arizona,,
42,West,3,Gonzaga,,,
48,Midwest,1,Michigan,Michigan,Michigan,Michigan
56,Midwest,6,Tennessee,,,


,Region,Seed,8,4,2,1
2,East,8,Ohio State,,,
8,East,6,Louisville,Louisville,,
16,South,1,Florida,Florida,Florida,
24,South,6,North Carolina,,,
38,West,4,Arkansas,,,
40,West,6,BYU,BYU,,
48,Midwest,1,Michigan,Michigan,Michigan,Michigan
62,Midwest,2,Iowa State,,,


## Initial Submissions

In [9]:
EXPORT_JSON = f"{DATA_DIRECTORY}/brackets_submit.json".format(gender=GENDER)
EXPORT_DIRECTORY = f"{DATA_DIRECTORY}/brackets_submit/".format(gender=GENDER)

In [10]:
%%time
with open(EXPORT_JSON, 'w') as fp:
    json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 3.77 ms, sys: 2.72 ms, total: 6.5 ms
Wall time: 26.1 ms


In [11]:
save_brackets(EXPORT_JSON, EXPORT_DIRECTORY, 'all', [f'bracket{i}' for i in range(10)])

## Prod Runs

In [12]:
%%time
# NUM_BRACKETS = 100
NUM_BRACKETS = 1000000
for gender in ["mens", "womens"]:
    initial_bracket, data = create_initial_bracket(DATA_FILENAME.format(gender=gender), ORDER_OF_REGIONS[gender])
    bracket_dict = parallel_predictions(NUM_BRACKETS, initial_bracket, data, sd_params, k, score_method="538")
    with open(BRACKETS_JSON.format(gender=gender), 'w') as fp:
        json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 1h 10min 37s, sys: 3min, total: 1h 13min 38s
Wall time: 2h 47min 10s


In [12]:
%%time
gender = "mens"
initial_bracket, data = create_initial_bracket(DATA_FILENAME.format(gender=gender), ORDER_OF_REGIONS[gender])
bracket_dict = parallel_predictions(1000000, initial_bracket, data, sd_params, k, score_method="kenpom")

with open(f"{DATA_DIRECTORY.format(gender='mens')}/brackets_kenpom.json", 'w') as fp:
    json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 37min 38s, sys: 1min 16s, total: 38min 54s
Wall time: 4h 1s


In [9]:
%%time
with open(f"{DATA_DIRECTORY}/brackets_2022_538.json", 'w') as fp:
    json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 15min 52s, sys: 32.7 s, total: 16min 25s
Wall time: 16min 26s


In [9]:
%%time
with open(f"{DATA_DIRECTORY}/brackets_2022_12_1.9.json", 'w') as fp:
    json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 16min, sys: 33 s, total: 16min 33s
Wall time: 16min 36s


In [10]:
%%time
with open(f"{DATA_DIRECTORY}/brackets_2022_11.json", 'w') as fp:
    json.dump(bracket_dict, fp, sort_keys=True, indent=4, cls=JSONEncoder)

CPU times: user 15min 7s, sys: 35.9 s, total: 15min 43s
Wall time: 15min 44s
